In [1]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from pyproj import Proj


In [2]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

SEQ_IN = 20
BATCH_SIZE = 64
EPOCHS = 40
LR = 1e-4

amax_knots = 0.15
amax = amax_knots * 0.514
proj = Proj(proj="utm", zone=31, ellps="WGS84")


In [3]:
def clean_ais(df):
    df = df.copy()

    df["BaseDateTime"] = pd.to_datetime(df["BaseDateTime"])
    df = df.sort_values(["MMSI", "BaseDateTime"])
    df['LAT'] = pd.to_numeric(df['LAT'], errors='coerce')
    df['LON'] = pd.to_numeric(df['LON'], errors='coerce')
    df['SOG'] = pd.to_numeric(df['SOG'], errors='coerce')
    df['COG'] = pd.to_numeric(df['COG'], errors='coerce')
    
    df = df.drop_duplicates(subset=["MMSI", "BaseDateTime"])
    df = df[df["SOG"] > 0]
    df = df[df["SOG"] <= 30]
    df = df[df["COG"].between(0, 360)]
    df = df[df["VesselType"].between(20, 90)]

    df["dt"] = df.groupby("MMSI")["BaseDateTime"].diff().dt.total_seconds()
    df = df[(df["dt"].isna()) | (df["dt"] >= 1)]

    return df.drop(columns=["dt"])


In [4]:
def extract_trajectories(df):
    trajs = []

    for mmsi, g in df.groupby("MMSI"):
        g = g.sort_values("BaseDateTime").copy()
        g["dt"] = g["BaseDateTime"].diff().dt.total_seconds()
        g["traj_id"] = (g["dt"] > 600).cumsum()

        for _, t in g.groupby("traj_id"):
            if len(t) < 24:
                continue
            duration = (t["BaseDateTime"].iloc[-1] - t["BaseDateTime"].iloc[0]).total_seconds() / 3600
            if duration > 24:
                continue
            if t["SOG"].mean() < 1:
                continue

            trajs.append(t.reset_index(drop=True))

    return trajs


In [5]:
def correct_speed(traj):
    sog = traj["SOG"].values.copy()
    t = traj["BaseDateTime"].values

    for i in range(1, len(traj)-1):
        dt1 = (t[i] - t[i-1]) / np.timedelta64(1, "s")
        dt2 = (t[i+1] - t[i]) / np.timedelta64(1, "s")
        if dt1 <= 0 or dt2 <= 0:
            continue

        if abs(sog[i] - sog[i-1]) > amax_knots * dt1:
            sog[i] = (dt1*sog[i-1] + dt2*sog[i+1]) / (dt1 + dt2)

    traj["SOG"] = sog
    return traj


In [6]:
def correct_position(traj):
    lat = traj["LAT"].values
    lon = traj["LON"].values
    t = traj["BaseDateTime"].values
    sog = traj["SOG"].values

    x, y = proj(lon, lat)

    for i in range(1, len(traj)-1):
        dt = (t[i] - t[i-1]) / np.timedelta64(1, "s")
        if dt <= 0:
            continue

        v = sog[i-1] * 0.514
        dmax = v*dt + 0.5*amax*(dt**2)
        d = math.hypot(x[i]-x[i-1], y[i]-y[i-1])

        if d > dmax:
            dt2 = (t[i+1] - t[i]) / np.timedelta64(1, "s")
            if dt2 <= 0:
                continue

            v2 = sog[i] * 0.514
            dmax2 = v2*dt2 + 0.5*amax*(dt2**2)
            d2 = math.hypot(x[i+1]-x[i], y[i+1]-y[i])

            if d2 > dmax2:
                alpha = (t[i]-t[i-1]) / (t[i+1]-t[i-1])
                x[i] = x[i-1] + alpha*(x[i+1]-x[i-1])
                y[i] = y[i-1] + alpha*(y[i+1]-y[i-1])

    lon_c, lat_c = proj(x, y, inverse=True)
    traj["LAT"] = lat_c
    traj["LON"] = lon_c
    return traj


In [7]:
def resample(traj):
    for col in ["LAT", "LON", "SOG", "COG"]:
        traj[col] = pd.to_numeric(traj[col], errors='coerce')

    # Resample et interpolation
    traj = traj.set_index("BaseDateTime").resample("1min").interpolate()
    x, y = proj(traj["LON"].values, traj["LAT"].values)
    dist = np.sqrt(np.diff(x)**2 + np.diff(y)**2)
    traj["SOG"] = np.insert(dist/60/0.514, 0, traj["SOG"].iloc[0])

    cog = np.unwrap(np.deg2rad(traj["COG"].values))
    traj["COG"] = np.rad2deg(cog) % 360

    return traj.reset_index()


In [8]:
def plot_before_after(raw_trajs, corr_trajs, n=5):
    plt.figure(figsize=(10,8))

    for i in range(n):
        plt.plot(raw_trajs[i]["LON"], raw_trajs[i]["LAT"], 'r--', alpha=0.5)

    for i in range(n):
        plt.plot(corr_trajs[i]["LON"], corr_trajs[i]["LAT"], 'b')

    plt.legend(["Raw", "Corrected"])
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.title("Trajectories Before vs After Correction")
    plt.grid(True)
    plt.show()


In [9]:
class AISTrajDataset(Dataset):
    def __init__(self, df):
        self.X, self.Y = [], []

        for _, g in df.groupby("MMSI"):
            lat, lon = g["LAT"].values, g["LON"].values
            sog = g["SOG"].values / 30
            cog = np.deg2rad(g["COG"].values)

            dx = (lat - lat[0]) * 111320
            dy = (lon - lon[0]) * 111320 * np.cos(np.deg2rad(lat[0]))

            feats = np.stack([dx, dy, sog, np.sin(cog), np.cos(cog)], axis=1)

            for i in range(len(feats)-SEQ_IN):
                self.X.append(feats[i:i+SEQ_IN])
                self.Y.append(feats[i+1:i+SEQ_IN+1])

        self.X = torch.tensor(self.X, dtype=torch.float32)
        self.Y = torch.tensor(self.Y, dtype=torch.float32)

    def __len__(self): return len(self.X)
    def __getitem__(self, i): return self.X[i], self.Y[i]


In [10]:
class LSTMTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        self.embed = nn.Linear(5, 128)
        self.lstm = nn.LSTM(128, 128, batch_first=True)
        enc = nn.TransformerEncoderLayer(128, 8, batch_first=True)
        self.tr = nn.TransformerEncoder(enc, 2)
        self.out = nn.Linear(128, 5)

    def forward(self, x):
        x = self.embed(x)
        x, _ = self.lstm(x)
        x = self.tr(x)
        return self.out(x)


In [11]:
def train(model, loader):
    opt = torch.optim.Adam(model.parameters(), LR)
    loss_fn = nn.MSELoss()
    model.train()

    for e in range(EPOCHS):
        tot = 0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad()
            loss = loss_fn(model(x), y)
            loss.backward()
            opt.step()
            tot += loss.item() * x.size(0)
        print(f"Epoch {e+1} | MSE {tot/len(loader.dataset):.6f}")


In [12]:
def mse(pred, true):
    return ((pred - true) ** 2).mean()


def rmse(pred, true):
    return torch.sqrt(mse(pred, true))


def haversine(lat1, lon1, lat2, lon2):
    R = 6371000.0
    phi1, phi2 = torch.deg2rad(lat1), torch.deg2rad(lat2)
    dphi = phi2 - phi1
    dl = torch.deg2rad(lon2 - lon1)
    a = torch.sin(dphi/2)**2 + torch.cos(phi1)*torch.cos(phi2)*torch.sin(dl/2)**2
    return 2 * R * torch.atan2(torch.sqrt(a), torch.sqrt(1-a))


def ade_fde(pred, true):
    ade = haversine(
        pred[:, :, 0], pred[:, :, 1],
        true[:, :, 0], true[:, :, 1]
    ).mean()

    fde = haversine(
        pred[:, -1, 0], pred[:, -1, 1],
        true[:, -1, 0], true[:, -1, 1]
    ).mean()

    return ade.item(), fde.item()

In [13]:
def evaluate(model, loader):
    model.eval()
    preds = []
    trues = []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            pred = model(x).cpu()
            preds.append(pred)
            trues.append(y)

    preds = torch.cat(preds)
    trues = torch.cat(trues)

    mse_v = mse(preds, trues).item()
    rmse_v = rmse(preds, trues).item()
    ade, fde = ade_fde(preds, trues)

    print(f"MSE  : {mse_v:.6f}")
    print(f"RMSE : {rmse_v:.4f}")
    print(f"ADE  : {ade:.2f} m")
    print(f"FDE  : {fde:.2f} m")

In [ ]:
df = pd.read_csv("AIS_2024_01_04.csv")
df = clean_ais(df)

raw_trajs = extract_trajectories(df)
corr_trajs = []

for t in raw_trajs:
    for _ in range(3):
        t = correct_speed(t)
        t = correct_position(t)
    corr_trajs.append(resample(t))

plot_before_after(raw_trajs, corr_trajs)

df_final = pd.concat(corr_trajs)
dataset = AISTrajDataset(df_final)
loader = DataLoader(dataset, BATCH_SIZE, shuffle=True)

model = LSTMTransformer().to(DEVICE)
train(model, loader)
evaluate(model, loader)

torch.save(model.state_dict(), "ais_lstm_transformer.pth")